In [1]:
from dotenv import find_dotenv, load_dotenv
from langchain.chains import ConversationalRetrievalChain, RetrievalQA
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from transformers import AutoTokenizer

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
CHROMA_STORAGE_PATH = "../data/vectors/chroma"

In [3]:
load_dotenv(find_dotenv("../../creds/.env"), verbose=True)

True

In [4]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
)

embedding = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

vector_db = Chroma(
    embedding_function=embedding,
    persist_directory=CHROMA_STORAGE_PATH,
)
vector_db._collection.count()

721

### Plain vector similarity

In [5]:
question = "What are the major topics for this class?"
docs = vector_db.similarity_search(
    question,
    k=3,
)
for doc in docs:
    print(f"=========== {doc.metadata['source'][-13:-4]}, page {doc.metadata['page']}\n", doc.page_content)

=========== Lecture04, page 0
 MachineLearning-Lecture04  
Instructor (Andrew Ng):Okay, good morning. Just a few administrative announcements 
before we jump into today’s technical material. So let’s see, by later today, I’ll post on 
the course website a handout with the sort of guidelines and suggestions for choosing and 
proposing class projects.  
So project proposals – so for the term project for this class due on Friday, the 19th of this 
month at noon – that’s about two weeks, two and a half weeks from now. If you haven’t 
yet formed teams or started thinking about project ideas, please do so.  
And later today, you’ll find on the course website a handout with the guidelines and some 
of the details on how to send me your proposals and so on.  
If you’re not sure whether an idea you have for a project may be a appropriate, or you’re 
sort of just fishing around for ideas or looking for ideas of projects to do, please, be 
strongly encouraged to come to my office hours on Friday 

### Vector + prompt

In [6]:
template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum. Keep the answer as concise as possible.
Always say "thanks for asking!" at the end of the answer.
{context}

Question: {question}

Helpful Answer:
"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
QA_CHAIN_PROMPT

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nIf you don\'t know the answer, just say that you don\'t know, don\'t try to make up an answer.\nUse three sentences maximum. Keep the answer as concise as possible.\nAlways say "thanks for asking!" at the end of the answer.\n{context}\n\nQuestion: {question}\n\nHelpful Answer:\n')

In [7]:
question = "Is probability a class topic?"
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": QA_CHAIN_PROMPT,
    },
)

result = qa_chain.invoke({"query": question})
result["result"].strip()

'Yes, probability is a key topic in this class, covered in lectures and discussion sections. The course builds on probability foundations for algorithms like logistic regression. The instructor emphasizes its importance for understanding empirical risk minimization. Thanks for asking!'

### Vector + prompt + memory

In [10]:
# langchain.debug = True


condense_template = """
Given the following conversation and a follow up question, rephrase the follow up question
to be a standalone question that incorporates all necessary context from the chat history, in its original language.

Chat History:
{chat_history}

Follow Up Input: {question}
Standalone question:
"""
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(condense_template)


question = "Who are the TAs?"

buffer_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=vector_db.as_retriever(),
    memory=buffer_memory,
    condense_question_prompt=CONDENSE_QUESTION_PROMPT,
)

result = qa_chain.invoke({"question": question})

print(result["answer"].strip())
result

The TAs mentioned in the context are:  

1. **Catie Chang** - A neuroscientist who applies machine learning algorithms to understand the human brain.  
2. **Tom Do** - A PhD student working in computational biology and the fundamentals of human learning.  
3. **Zico Kolter** - The head TA (for two consecutive years) who applies machine learning to robotics.  
4. **Daniel Ramage** - Applies learning algorithms to problems in natural language processing (though he is not present during the described period).  

These TAs work in interdisciplinary fields, including computer vision, biology, robotics, and language processing.


{'question': 'Who are the TAs?',
 'chat_history': [HumanMessage(content='Who are the TAs?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='\n\nThe TAs mentioned in the context are:  \n\n1. **Catie Chang** - A neuroscientist who applies machine learning algorithms to understand the human brain.  \n2. **Tom Do** - A PhD student working in computational biology and the fundamentals of human learning.  \n3. **Zico Kolter** - The head TA (for two consecutive years) who applies machine learning to robotics.  \n4. **Daniel Ramage** - Applies learning algorithms to problems in natural language processing (though he is not present during the described period).  \n\nThese TAs work in interdisciplinary fields, including computer vision, biology, robotics, and language processing.', additional_kwargs={}, response_metadata={})],
 'answer': '\n\nThe TAs mentioned in the context are:  \n\n1. **Catie Chang** - A neuroscientist who applies machine learning algorithms to understand th

In [11]:
question = "What are their majors?"
result = qa_chain.invoke({"question": question})

print(result["answer"].strip())
result

The TAs mentioned in the context have the following majors or areas of focus:  

1. **Catie Chang**: Neuroscientist (applies machine learning to understand the human brain).  
2. **Tom Do**: Computational biology (works on the fundamentals of human learning).  
3. **Zico Kolter**: Machine learning (applies algorithms to robotics).  
4. **Daniel Ramage**: Natural language processing (applies learning algorithms to language problems).  

These TAs work across interdisciplinary fields, combining machine learning with domains like neuroscience, biology, robotics, and linguistics.


{'question': 'What are their majors?',
 'chat_history': [HumanMessage(content='Who are the TAs?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='\n\nThe TAs mentioned in the context are:  \n\n1. **Catie Chang** - A neuroscientist who applies machine learning algorithms to understand the human brain.  \n2. **Tom Do** - A PhD student working in computational biology and the fundamentals of human learning.  \n3. **Zico Kolter** - The head TA (for two consecutive years) who applies machine learning to robotics.  \n4. **Daniel Ramage** - Applies learning algorithms to problems in natural language processing (though he is not present during the described period).  \n\nThese TAs work in interdisciplinary fields, including computer vision, biology, robotics, and language processing.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What are their majors?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='\n\nThe TAs mentioned in the cont